# Development 1: persistent storage, token and session

Run top to bottom at the start of every Colab session. The runtime disk is
wiped between sessions, so nothing here assumes a file from last time.

Your code reaches the runtime through cell 2, which carries a packed copy of
`src/`, `cpp/`, `tests/` and `pyproject.toml`. After editing code locally, refresh it:

    python -m vggt_aura.sync

then re-run cell 2. No git and no GitHub are involved. (Notebooks are committed with that cell EMPTY, so the
command above is also the first thing to run after cloning the repository.)

The only secret needed is `HF_TOKEN`. Keep it in a `.env` file on Google Drive at
`MyDrive/vggt-omega-aura-benchmark/.env` (one line: `HF_TOKEN=...`). It survives sessions,
so nothing needs typing. Colab secrets time out through the VS Code extension and are not used.

In [1]:
# --- 1. Configuration: the only cell to edit ---
PERSIST_MODE = "drive"     # verified 2026-09-18: Drive mounts through the VS Code extension. "runtime" is the fallback
REQUIRE_GPU = False        # set True for model inference (`03_first_forward_pass` onwards). Download and inspection need no GPU
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"

In [2]:
# --- 1b. Fail fast: is a GPU attached? Then mount Drive (results and the token live there) ---
import subprocess
from pathlib import Path

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    print("GPU:", gpu.stdout.strip())
elif REQUIRE_GPU:
    raise RuntimeError("No GPU attached. Change the runtime type to a GPU (A100 if offered), reconnect, and re-run.")
else:
    print("No GPU attached, continuing because REQUIRE_GPU is False")

if PERSIST_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

GPU: Tesla T4, 15360 MiB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [4]:
# --- 3. Upstream packages at pinned commits (public, no token needed) ---
import subprocess, sys
from vggt_aura import pins


def pip(*args):
    print("pip install", *args)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


# --no-deps: upstream pins numpy<2, which would downgrade NumPy under the running kernel.
pip("--no-deps", f"git+{pins.VGGT_OMEGA_REPO}@{pins.VGGT_OMEGA_COMMIT}")
pip("einops", "safetensors", "pybind11")  # pybind11 is not preinstalled on Colab
pip(f"fzi-aura[download] @ git+{pins.AURA_SDK_REPO}@{pins.AURA_SDK_COMMIT}")

pip install --no-deps git+https://github.com/facebookresearch/vggt-omega@a3ab0141f96838724423541044ff5ba301cfd36a
pip install einops safetensors pybind11
pip install fzi-aura[download] @ git+https://github.com/fzi-forschungszentrum-informatik/fzi-aura-sdk@a36761db6c5ec4ad6e30064dce63a4437340a513


In [5]:
# --- 4. Hugging Face token: environment, then .env on Drive, then /content/.env, then a hidden prompt ---
import os
from pathlib import Path
from vggt_aura.credentials import load_secret

os.environ["HF_TOKEN"] = load_secret("HF_TOKEN", env_files=[Path(DRIVE_ROOT) / ".env", Path("/content/.env")], prompt=True)
from huggingface_hub import whoami
print("Hugging Face user:", whoami()["name"])

Hugging Face user: saverino


In [6]:
# --- 5. Persist root and resume manifest ---
from vggt_aura.persistence import Manifest, resolve_persist_root

PERSIST_ROOT = resolve_persist_root(PERSIST_MODE, project_root=PROJECT_DIR, drive_root=DRIVE_ROOT)
manifest = Manifest(PERSIST_ROOT / "manifest" / "completed_scenes.jsonl")
print("persist root:", PERSIST_ROOT)
for stage in ("predict", "ground_truth", "metrics"):
    print(f"  {stage}: {len(manifest.done(stage))} scenes already done")
if PERSIST_MODE == "runtime":
    print("WARNING: runtime mode does not survive the session. Run the last cell and download the zip before disconnecting.")

persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
  predict: 0 scenes already done
  ground_truth: 0 scenes already done
  metrics: 0 scenes already done


In [7]:
# --- 6. Toolchain report, and the C++ build once cpp/ has a CMakeLists.txt (see `06_cpp_core`) ---
for tool in ("g++ --version", "cmake --version"):
    result = subprocess.run(tool, shell=True, capture_output=True, text=True)
    print(tool.split()[0], "->", result.stdout.splitlines()[0] if result.returncode == 0 else "MISSING")
try:
    import pybind11
    print("pybind11 ->", pybind11.__version__)
except ImportError:
    print("pybind11 -> MISSING")

from vggt_aura import cpp_build   # knows where pybind11 lives, and falls back to a direct compiler command
print("C++ core:", "built and importable" if cpp_build.build(PROJECT_DIR) else "not built (Python stays the reference)")

g++ -> g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
cmake -> cmake version 3.31.10
pybind11 -> 3.1.0
cpp/ has no CMakeLists.txt yet, nothing to build


## End of run (runtime mode only)

Run the cell below, then use the VS Code command `Colab: Download...` to
fetch `/content/results.zip`, and unzip it over the local `results/` folder.
In drive mode the outputs are already on Drive and this is not needed.

In [8]:
# --- 7. Pack results for download (runtime mode only; safe to run in any mode) ---
import shutil

if PERSIST_MODE == "drive":
    print("drive mode: results are already on Drive at", PERSIST_ROOT, "- nothing to pack")
elif not (PROJECT_DIR / "results").is_dir():
    print("no results folder on the runtime yet - nothing to pack")
else:
    archive = shutil.make_archive("/content/results", "zip", root_dir=PROJECT_DIR, base_dir="results")
    print("ready to download:", archive, f"({Path(archive).stat().st_size / 1e6:.2f} MB)")

drive mode: results are already on Drive at /content/drive/MyDrive/vggt-omega-aura-benchmark - nothing to pack
